In [37]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/FOIA_7a_FY2010_FY2019_asof_260630.csv"
PROCESSED_PATH = "../data/processed/sba_7a_cleaned.csv"

df = pd.read_csv(RAW_PATH, low_memory=False)

df.shape

(545751, 42)

In [38]:
clean_df = df[df["LoanStatus"].isin(["P I F", "CHGOFF"])].copy()

clean_df["default"] = clean_df["LoanStatus"].map({
    "P I F": 0,
    "CHGOFF": 1
})

clean_df["default"].value_counts(normalize=True).mul(100).round(2)

default
0    92.04
1     7.96
Name: proportion, dtype: float64

In [39]:
clean_df["sba_guarantee_ratio"] = (
    clean_df["SBAGuaranteedApproval"] / clean_df["GrossApproval"]
)

clean_df["is_franchise"] = clean_df["FranchiseCode"].notna().astype(int)

clean_df["naics_sector"] = (
    clean_df["NaicsCode"]
    .astype("Int64")
    .astype(str)
    .str[:2]
)

In [40]:
date_cols = ["ApprovalDate", "FirstDisbursementDate"]

for col in date_cols:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

In [41]:
clean_df["approval_month"] = clean_df["ApprovalDate"].dt.month
clean_df["approval_quarter"] = clean_df["ApprovalDate"].dt.quarter

In [42]:
clean_df["sold_secondary_market"] = clean_df["SoldSecMrktInd"].eq("Y").astype(int)

In [43]:
drop_cols = [
    "LoanStatus",
    "PaidInFullDate",
    "ChargeOffDate",
    "GrossChargeOffAmount",
    "AsOfDate",
    "Program",
    "LocationID",
    "BorrName",
    "BorrStreet",
    "BorrCity",
    "BorrZip",
    "BankName",
    "BankFDICNumber",
    "BankNCUANumber",
    "BankStreet",
    "BankCity",
    "BankZip",
    "FranchiseCode",
    "FranchiseName",
    "NaicsDescription",
    "SoldSecMrktInd",
    "ProjectCounty",
    "ApprovalDate",
    "FirstDisbursementDate",
]

In [44]:
clean_df = clean_df.drop(columns=drop_cols)

clean_df.shape

(428874, 25)

In [45]:
clean_df.loc[clean_df["InitialInterestRate"] == 0, "InitialInterestRate"] = np.nan
clean_df.loc[clean_df["TermInMonths"] == 0, "TermInMonths"] = np.nan

In [46]:
missing_after_drop = (
    clean_df
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

missing_after_drop[missing_after_drop > 0]

BusinessAge              0.29
TermInMonths             0.05
CongressionalDistrict    0.01
dtype: float64

In [47]:
numeric_cols = clean_df.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_cols = clean_df.select_dtypes(include=["object", "string"]).columns.tolist()

numeric_cols.remove("default")

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['GrossApproval', 'SBAGuaranteedApproval', 'ApprovalFY', 'InitialInterestRate', 'TermInMonths', 'NaicsCode', 'CongressionalDistrict', 'JobsSupported', 'sba_guarantee_ratio', 'is_franchise', 'sold_secondary_market']
Categorical: ['BorrState', 'BankState', 'ProcessingMethod', 'FixedorVariableInterestInd', 'ProjectState', 'SBADistrictOffice', 'BusinessType', 'BusinessAge', 'RevolverStatus', 'CollateralInd', 'naics_sector']


In [48]:
clean_df.head()

,BorrState,BankState,GrossApproval,SBAGuaranteedApproval,ApprovalFY,ProcessingMethod,InitialInterestRate,FixedorVariableInterestInd,TermInMonths,NaicsCode,...,RevolverStatus,JobsSupported,CollateralInd,default,sba_guarantee_ratio,is_franchise,naics_sector,approval_month,approval_quarter,sold_secondary_market
0,TX,TX,288000.0,259200.0,2010,Preferred Lenders Program,6.00,V,120.0,722110.0,...,N,18.0,Y,0,0.9,0,72,10,4,0
1,FL,NC,1200000.0,1080000.0,2010,Preferred Lenders Program,4.75,V,300.0,541940.0,...,N,10.0,N,0,0.9,0,54,10,4,1
2,TX,OH,120000.0,108000.0,2010,Preferred Lenders Program,5.25,V,90.0,312113.0,...,N,4.0,N,0,0.9,0,31,10,4,0
3,MI,OH,150000.0,75000.0,2010,SBA Express Program,5.25,V,60.0,722211.0,...,N,48.0,Y,0,0.5,1,72,10,4,0
5,NY,OH,25000.0,12500.0,2010,SBA Express Program,6.50,V,84.0,238210.0,...,Y,4.0,Y,0,0.5,0,23,10,4,0


In [49]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 428874 entries, 0 to 545749
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   BorrState                   428874 non-null  str    
 1   BankState                   428874 non-null  str    
 2   GrossApproval               428874 non-null  float64
 3   SBAGuaranteedApproval       428874 non-null  float64
 4   ApprovalFY                  428874 non-null  int64  
 5   ProcessingMethod            428874 non-null  str    
 6   InitialInterestRate         428869 non-null  float64
 7   FixedorVariableInterestInd  428874 non-null  str    
 8   TermInMonths                428652 non-null  float64
 9   NaicsCode                   428872 non-null  float64
 10  ProjectState                428874 non-null  str    
 11  SBADistrictOffice           428874 non-null  str    
 12  CongressionalDistrict       428849 non-null  float64
 13  BusinessType                42

In [50]:
clean_df["default"].value_counts(normalize=True).mul(100).round(2)

default
0    92.04
1     7.96
Name: proportion, dtype: float64

In [51]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)

In [52]:
pd.read_csv(PROCESSED_PATH).shape

(428874, 25)

## Cleaning Decisions

- Kept only clear final outcomes: `P I F` and `CHGOFF`
- Created binary target `default`
- Removed leakage columns known after loan outcome
- Removed identity, name, address, and high-cardinality fields
- Removed `Program` because all rows are SBA 7(a)
- Replaced raw franchise fields with `is_franchise`
- Created `sba_guarantee_ratio`
- Created `naics_sector`
- Replaced raw `SoldSecMrktInd` with `sold_secondary_market`
- Extracted approval month and quarter from `ApprovalDate`
- Left remaining missing values for model pipeline imputation